# Default clustering with clustbench

If you have data `X` of shape `(n_samples, n_features)` and you don't already
know which clustering algorithm is right for it, this notebook is the
recommended starting point. The default algorithm is
[`learned_router_v3`](../src/clustbench/algorithms/learned_router_v3.py) — a
meta-algorithm that fingerprints your data and dispatches to whichever of
the registered base clusterers is predicted to score best on that
fingerprint. Across nine rounds of benchmark iteration documented in
[`docs/ALGORITHM_ANALYSIS.md`](../docs/ALGORITHM_ANALYSIS.md), it landed at
or near rank 1 with mean ARI 0.839 across 55 dataset tasks (time-series,
graph nodes, 23 synthetic families, and real-world UCI sets), which makes
it the empirical answer to *"which clustering algorithm should I use when
I don't know my data's regime?"*

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt

# Importing the package triggers @register on every algorithm module,
# which populates ALGO_REGISTRY (used in the "trust but verify" sweep below).
import clustbench  # noqa: F401
from clustbench.algorithms.base import ALGO_REGISTRY
from clustbench.algorithms.learned_router_v3 import Learned_router_v3
from clustbench.datasets import DataSpec, gen_mdcgen, gen_iris  # noqa: F401

warnings.filterwarnings("ignore")
print(f"{len(ALGO_REGISTRY)} algorithms registered")

## Step 1: prepare your data

`Learned_router_v3.fit_predict` takes a 2-D float array `X` of shape
`(n_samples, n_features)` and a target number of clusters `k`. There are
no other shape requirements — no labels, no graph, no precomputed
distances. For this walkthrough we generate a small synthetic mixture
with `gen_mdcgen` so we have ground-truth labels to evaluate against;
on your own data, skip this cell and load your features directly into `X`.

In [ ]:
spec = DataSpec(
    n_samples=400,
    n_features=10,
    centers=3,           # DataSpec's `centers` is the `k` from the brief.
    compactness=1.0,
    seed=42,
    outliers=40,
)
X, y_true = gen_mdcgen(spec)
X = np.asarray(X, dtype=np.float64)

print("X.shape       :", X.shape)
print("unique y_true :", np.unique(y_true).tolist(),
      " (-1 == injected outliers)")

## Step 2: run the recommended dispatcher

Instantiate `Learned_router_v3` and call `fit_predict(X, k=...)`. The
result is an `AlgoResult` (see `clustbench.algorithms.base.AlgoResult`)
with three fields:

- `labels` — integer cluster ids, shape `(n_samples,)`.
- `extra["chose"]` — the name of the underlying algorithm that was
  dispatched to.
- `extra["fingerprint"]` — the 17-dimensional summary the router built
  from your data to make the choice.

In [ ]:
router = Learned_router_v3()
res = router.fit_predict(X, k=3)
labels = res.labels

print("chose      :", res.extra["chose"])
print("first 20   :", labels[:20].tolist())
print("fingerprint:")
for name, value in res.extra["fingerprint"].items():
    print(f"  {name:35s} {value:.4f}")

## Step 3: evaluate

When ground-truth labels are available, the standard external-validity
score for clustering is the **Adjusted Rand Index (ARI)** — chance-
corrected, symmetric in label permutations, and bounded in `[-1, 1]`
with `1.0` meaning the partitions match exactly. Below we compute ARI
on the non-outlier points (the injected outliers carry label `-1`) and
then project the data to 2D with PCA so we can see the ground truth
and the router's partition side by side.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

mask = y_true >= 0
ari = adjusted_rand_score(y_true[mask], labels[mask])
print(f"ARI vs ground truth: {ari:.3f}")

XY = PCA(n_components=2, random_state=0).fit_transform(X)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(XY[:, 0], XY[:, 1], c=y_true, s=10, cmap="tab10")
axes[0].set_title("Ground truth (-1 = outlier)")
axes[1].scatter(XY[:, 0], XY[:, 1], c=labels, s=10, cmap="tab10")
axes[1].set_title(f"Predicted by {res.extra['chose']} (ARI={ari:.3f})")
for ax in axes:
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
fig.tight_layout()
plt.show()

# Without ground truth, the standard internal-validity metric is silhouette.
from sklearn.metrics import silhouette_score
if len(set(labels)) > 1:
    print(f"silhouette : {silhouette_score(X, labels):.3f}")

## What if I don't have ground truth?

Most real datasets don't come with labels. In that case the standard
**internal-validity** metric is the **silhouette coefficient** — the
average of `(b - a) / max(a, b)` per point, where `a` is the mean
intra-cluster distance and `b` the mean distance to the nearest other
cluster. Bounded in `[-1, 1]`, higher is better. It rewards tight,
well-separated clusters and is defined whenever you have more than one
cluster. The snippet at the bottom of the previous cell is the one-
liner you'd use:

```python
from sklearn.metrics import silhouette_score
silhouette_score(X, labels)
```

In [ ]:
# Trust but verify: rerun every base clusterer in the registry on the same
# data and rank by ARI. The router's pick should sit near the top. We skip
# the other meta-algorithms so the comparison reflects base clusterers only.
import time

SKIP_PREFIXES = (
    "learned_router", "meta_clusterer", "mutant_kmeans_meta",
    "consensus", "pwcc",
)

rows = []
for name in sorted(ALGO_REGISTRY):
    if name.startswith(SKIP_PREFIXES):
        continue
    cls = ALGO_REGISTRY[name]
    t0 = time.perf_counter()
    try:
        out = cls().fit_predict(X, k=3)
        cand = np.asarray(out.labels)
        if cand.shape[0] != X.shape[0]:
            raise ValueError(f"label length mismatch: {cand.shape}")
        a = adjusted_rand_score(y_true[mask], cand[mask])
        rows.append((name, a, time.perf_counter() - t0))
    except Exception:
        rows.append((name, float("nan"), time.perf_counter() - t0))

# Add the router's own result for the comparison.
rows.append((f"learned_router_v3 -> {res.extra['chose']}", ari, float("nan")))

rows.sort(key=lambda r: (-(r[1] if not np.isnan(r[1]) else -np.inf), r[0]))
print(f"{'algorithm':40s}  {'ARI':>6s}  {'time_s':>7s}")
print("-" * 58)
for name, a, dt in rows:
    a_str = f"{a:.3f}" if not np.isnan(a) else "  nan"
    dt_str = f"{dt:.2f}" if not np.isnan(dt) else "   - "
    print(f"{name:40s}  {a_str:>6s}  {dt_str:>7s}")

## Where to go next

- **Algorithm analysis.** Each registered algorithm gets a per-axis
  diagnosis (scalability, clean-data ARI, outlier behaviour, shape
  adaptability) in [`docs/ALGORITHM_ANALYSIS.md`](../docs/ALGORITHM_ANALYSIS.md).
- **Methodology.** The recipe used to *find* `learned_router_v3` —
  synthesise from primitives that win on each axis, benchmark, fix
  the new failure mode, dispatch, then learn the dispatch rule — is in
  [`docs/METHODOLOGY.md`](../docs/METHODOLOGY.md). Follow it to retrain
  the router on a domain-specific benchmark of your own.
- **Benchmark configs.** Reproducible YAML grids live under
  [`configs/`](../configs/) — start with `benchmark.sample.yaml` for a
  smoke run, then `benchmark.paper.demo.yaml` for the full dashboard.
- **README.** The top-level [`README.md`](../README.md) ties these
  threads together and documents the JSON-over-stdin protocol for
  plugging non-Python clusterers into the same benchmark.